# HCHO (formaldéhyde) Trend Detective – Maroc (Google Colab)
**NASA Space Apps Challenge 2026 – Be An Earth System Trend Detective**

Colonne totale de **formaldéhyde (HCHO)** mesurée par **OMI/Aura** (`OMHCHOd`, journalier, V003, 0,1°), pour le **Maroc**, d'octobre 2004 à 2026, avec l'axe du temps divisé **tous les 6 mois** :
2004 = octobre → décembre (S2, début des données OMI), puis chaque année S1 = janvier → juin et S2 = juillet → décembre.

Chaque mois = moyenne **pondérée** (`sample_weight`) des jours valides (`data_quality_flag` = 0), regroupée en cellules de **0,5°** pour réduire le bruit. Unité : **10¹⁵ molécules/cm²**.

**Mode d'emploi :** exécutez les cellules **dans l'ordre**, de haut en bas. Chaque étape est dans sa propre cellule ; les 4 graphiques (étapes 5 à 8) se lancent chacun seul une fois les étapes 1 à 4 faites.

| Étape | Rôle |
|---|---|
| 1 | Recherche des fichiers Earthdata (identifiants demandés ici, jamais écrits dans le notebook) |
| 2 | Inspection de la structure d'un fichier |
| 3 | Extraction de la fenêtre du Maroc, moyennes mensuelles, unités |
| 4 | Semestres, anomalies, tendances |
| 5–8 | Les 4 graphiques : cartes, série temporelle, cycle saisonnier, anomalies |
| 9 | Export CSV |
| 10 | Synthèse : quoi / où / combien / significatif |
| 11 | Données du site web (`donnees_hcho.js`) |

> - **Étape 3 longue** : environ 8 000 fichiers journaliers (1 par jour). Seule la fenêtre du Maroc est lue dans chaque fichier, mais comptez **1 à 3 heures**. L'extraction est **reprenable** : si Colab se déconnecte, relancez les cellules de préparation puis l'étape 3, elle continue où elle s'était arrêtée. `UTILISER_DRIVE = True` (par défaut ici) garde le travail dans Google Drive.
> - Pour un premier essai rapide, mettez par exemple `DATE_FIN = "2006-12-31"` dans `config.py`.
> - Au Maroc, le HCHO est faible (fond de quelques 10¹⁵ molécules/cm²), proche de la limite de détection d'OMI : les tendances locales sont à interpréter avec prudence. La couverture d'OMI diminue après 2007 (« row anomaly »), ce qui augmente le bruit.
> - La moyenne est calculée sur les **cellules situées dans les frontières du Maroc** (Natural Earth). Une seule période de tendance par défaut (`FIN_PRE = None` ; mettez une date pour comparer deux périodes).

## Préparation

In [ ]:
!pip -q install earthaccess pymannkendall cartopy h5py netCDF4 geopandas shapely

In [ ]:
# Dossier de travail. UTILISER_DRIVE = True garde données et résultats dans Google Drive
# (sinon tout est perdu à la fermeture de Colab).
UTILISER_DRIVE = True
import os, sys
if UTILISER_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DOSSIER = "/content/drive/MyDrive/hcho_trend_detective"
else:
    DOSSIER = "/content/hcho_trend_detective"
os.makedirs(DOSSIER, exist_ok=True)
os.chdir(DOSSIER)
if DOSSIER not in sys.path:
    sys.path.insert(0, DOSSIER)
print("Dossier de travail :", os.getcwd())

## Fichiers communs
Les trois cellules suivantes écrivent `config.py` (tous les paramètres), `utils.py` (lecture, frontières du Maroc, statistiques) et `traces.py` (aides de tracé). Pour changer un paramètre, modifiez `config.py` ci-dessous, réexécutez cette cellule, puis lancez `import importlib, config; importlib.reload(config)` et réexécutez les étapes concernées (4 à 11 en général ; 3 aussi si vous changez la fenêtre, `AGREGAT` ou la qualité – supprimez alors le dossier `data/mois`).

In [ ]:
%%writefile config.py
"""Paramètres du projet – tout ce qu'on peut vouloir modifier est ici."""
from pathlib import Path

RACINE = Path(__file__).resolve().parent
DATA_BRUT = RACINE / "data" / "brut"        # index des fichiers NASA, frontières
DATA_MOIS = RACINE / "data" / "mois"        # 1 petit fichier par mois (fenêtre du Maroc) – extraction reprenable
DATA_TRAITE = RACINE / "data" / "traite"    # données intermédiaires
SORTIE = RACINE / "resultats"               # graphiques (PNG) et fichiers CSV
SITE = RACINE / "site"                      # données prêtes pour le site web
for _d in (DATA_BRUT, DATA_MOIS, DATA_TRAITE, SORTIE, SITE / "data" / "csv"):
    _d.mkdir(parents=True, exist_ok=True)

# --- Variable étudiée
NOM = "hcho"                   # identifiant (noms de dossiers et fichiers)
TITRE = "Formaldéhyde (HCHO)"               # nom affiché sur les graphiques
UNITE = "10¹⁵ molécules/cm²"

# --- Données NASA Earthdata
PRODUIT = "OMHCHOd"           # OMI/Aura, colonne totale de HCHO, journalier, 0,1°
VERSION = "003"
DATE_DEBUT = "2004-10-01"
DATE_FIN = "2026-12-31"

# --- Lecture des fichiers (None = choix automatique, voir étape 2)
VARIABLE = None                # chemin exact dans le fichier, ex. "key_science_data/column_amount"
MOTS_VARIABLE = ["column_amount"]  # mots-clés pour le choix automatique
EXCLURE = ["uncertainty"]
MOTS_POIDS = ["sample_weight"]        # poids de chaque cellule (moyenne pondérée), [] = aucun
MOTS_QUALITE = ["data_quality_flag"]    # indicateur de qualité, [] = aucun
QUALITE_OK = [0]        # valeurs de l'indicateur conservées
VALEUR_MIN = None        # valeurs plus petites rejetées (None = aucune)
CONVERSION = "facteur"      # "mm/mois" : taux de pluie -> cumul mensuel ; sinon multiplication par FACTEUR
FACTEUR = 1e-15
AGREGAT = 5              # regroupe AGREGAT x AGREGAT cellules de 0,1° (5 -> 0,5° : le HCHO journalier est très bruité)
PARALLELE = 8                  # fichiers lus en même temps

# --- Zone d'étude
ZONE = "Maroc"
INCLURE_SAHARA = True          # False = Maroc sans les provinces du Sud
ETENDUE = (-20, 2, 19, 38)     # fenêtre extraite et cartographiée : (lon_min, lon_max, lat_min, lat_max)

# --- Analyse
ALPHA = 0.05                   # seuil de significativité
Z_SEUIL = 1.5                  # seuil « semestre exceptionnel » (écarts-types)
MOIS_MIN_SEMESTRE = 3          # semestres avec moins de mois écartés (ex. dernier semestre en cours)
FIN_PRE = None             # fin de la 1re période comparée (None = une seule période)
NOM_COUPURE = ""

# --- Couleurs
CMAP_MOYENNE = "YlOrRd"
CMAP_TENDANCE = "RdBu_r"

# --- Affichage
AFFICHER = True                # False = enregistrer les images sans ouvrir de fenêtre

SORTIE_ZONE = SORTIE / NOM
SORTIE_ZONE.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile utils.py
"""Fonctions partagées par toutes les étapes (lecture, Maroc, semestres, statistiques)."""
import os
import tempfile
import time
from concurrent.futures import ProcessPoolExecutor
from types import SimpleNamespace
from urllib.request import urlretrieve

import h5py
import numpy as np
import pandas as pd
import pymannkendall as mk
import xarray as xr
from scipy import stats

import config as cfg


# =====================================================================
# Lecture des fichiers NASA (HDF5 / netCDF4) – seulement la fenêtre du Maroc
# =====================================================================
def lister(f):
    """Tous les tableaux du fichier : {chemin: dataset}."""
    sortie = {}
    f.visititems(lambda n, o: sortie.__setitem__(n, o) if isinstance(o, h5py.Dataset) else None)
    return sortie


def trouver(f, mots, exclure=()):
    """Chemin du premier tableau 2D+ dont le nom contient un des mots (nom exact prioritaire)."""
    if not mots:
        return None
    cand = [p for p, d in lister(f).items() if d.ndim >= 2
            and any(m in p.split("/")[-1].lower() for m in mots)
            and not any(e in p.lower() for e in exclure)]
    exact = [p for p in cand if p.split("/")[-1].lower() in mots]
    return (exact or cand or [None])[0]


def coordonnee(f, noms):
    for p, d in lister(f).items():
        if d.ndim == 1 and p.split("/")[-1].lower() in noms:
            return np.asarray(d[:], dtype=float)
    raise KeyError(noms)


def _attr(d, nom):
    v = d.attrs.get(nom)
    if v is None:
        return None
    v = v.decode() if isinstance(v, bytes) else v
    v = v[0] if isinstance(v, np.ndarray) and v.size == 1 else v
    return v.decode() if isinstance(v, bytes) else v


def lire_fenetre(f, chemin, etendue):
    """Lit chemin dans la fenêtre (lon_min, lon_max, lat_min, lat_max) -> tableau (lat, lon)."""
    lat = coordonnee(f, ("lat", "latitude"))
    lon = coordonnee(f, ("lon", "longitude"))
    lo0, lo1, la0, la1 = etendue
    il = np.where((lat >= la0) & (lat <= la1))[0]
    jl = np.where((lon >= lo0) & (lon <= lo1))[0]
    sl_la, sl_lo = slice(il.min(), il.max() + 1), slice(jl.min(), jl.max() + 1)
    d = f[chemin]
    idx, axes = [], []
    for n in d.shape:                          # axes repérés par leur longueur
        if n == len(lat) and "lat" not in axes:
            idx.append(sl_la); axes.append("lat")
        elif n == len(lon) and "lon" not in axes:
            idx.append(sl_lo); axes.append("lon")
        else:
            idx.append(0)                      # axe du temps (1 seul pas)
    a = np.asarray(d[tuple(idx)], dtype="float64")
    if axes == ["lon", "lat"]:
        a = a.T
    for nom in ("_FillValue", "missing_value"):
        fv = _attr(d, nom)
        if fv is not None:
            a[a == float(fv)] = np.nan
    a[np.abs(a) > 1e29] = np.nan
    sf, off = _attr(d, "scale_factor"), _attr(d, "add_offset")
    if sf is not None:
        a = a * float(sf)
    if off is not None:
        a = a + float(off)
    la, lo = lat[sl_la], lon[sl_lo]
    if la[0] > la[-1]:
        a, la = a[::-1], la[::-1]
    return a, la, lo, str(_attr(d, "units") or _attr(d, "Units") or "")


def lire_granule(source):
    """Un fichier -> (valeur, poids, lat, lon, unité) sur la fenêtre de la zone."""
    with h5py.File(source, "r") as f:
        var = cfg.VARIABLE or trouver(f, cfg.MOTS_VARIABLE, cfg.EXCLURE)
        a, la, lo, unite = lire_fenetre(f, var, cfg.ETENDUE)
        w = np.ones_like(a)
        p = trouver(f, cfg.MOTS_POIDS)
        if p:
            w = np.nan_to_num(lire_fenetre(f, p, cfg.ETENDUE)[0], nan=0.0)
        q = trouver(f, cfg.MOTS_QUALITE)
        if q and cfg.QUALITE_OK is not None:
            qa = lire_fenetre(f, q, cfg.ETENDUE)[0]
            a[~np.isin(qa, cfg.QUALITE_OK)] = np.nan
    if cfg.VALEUR_MIN is not None:
        a[a < cfg.VALEUR_MIN] = np.nan
    return a, w, la, lo, unite


# --- accès aux fichiers : chemin local, sinon lecture partielle à distance (Earthdata), sinon téléchargement
_FS = None


def preparer_acces():
    """Connexion Earthdata (identifiants enregistrés dans ~/.netrc pour les processus de lecture)."""
    import earthaccess
    earthaccess.login(persist=True)


def _fs():
    global _FS
    if _FS is None:
        import earthaccess
        if os.environ.get("EARTHDATA_USERNAME") and os.environ.get("EARTHDATA_PASSWORD"):
            earthaccess.login(strategy="environment")
        else:
            earthaccess.login(strategy="netrc")
        _FS = earthaccess.get_fsspec_https_session()
    return _FS


def _init_processus():
    global _FS
    _FS = None                    # chaque processus ouvre sa propre connexion


def lire_url(url, essais=3):
    """Lit la fenêtre d'un fichier ; None si le fichier reste illisible."""
    if os.path.exists(url):
        return lire_granule(url)
    err = None
    for k in range(essais):
        try:
            with _fs().open(url, "rb", block_size=4 * 2 ** 20) as fo:   # lecture partielle (fenêtre seulement)
                return lire_granule(fo)
        except Exception as e:
            err = e
            time.sleep(2 * (k + 1))
    try:                                                              # dernier recours : téléchargement complet
        import earthaccess
        with tempfile.TemporaryDirectory() as tmp:
            chemin = earthaccess.download([url], tmp, show_progress=False)[0]
            return lire_granule(chemin)
    except Exception as e:
        print(f"   ! échec {url.split('/')[-1]} : {err} / {e}", flush=True)
        return None


def processus():
    """Lecteurs parallèles (processus séparés : la bibliothèque HDF5 n'accepte pas les threads)."""
    return ProcessPoolExecutor(cfg.PARALLELE, initializer=_init_processus)


def agreger(x, k):
    """Somme par blocs k x k (réduction de résolution)."""
    if k == 1:
        return x
    ny, nx = (x.shape[0] // k) * k, (x.shape[1] // k) * k
    return x[:ny, :nx].reshape(ny // k, k, nx // k, k).sum((1, 3))


def enregistrer_mois(mois, resultats, n_fichiers):
    """Moyenne pondérée des fichiers d'un mois -> data/mois/<mois>.nc (False si rien de lisible)."""
    s = sw = nj = None
    la = lo = unite = None
    for r in resultats:
        if r is None:
            continue
        a, w, la, lo, unite = r
        ok = np.isfinite(a) & (w > 0)
        if s is None:
            s, sw, nj = np.zeros_like(a), np.zeros_like(a), np.zeros_like(a)
        s += np.where(ok, a * w, 0)
        sw += np.where(ok, w, 0)
        nj += ok
    if s is None:
        print(f"   {mois} : aucun fichier lisible – relancer l'étape 3 plus tard", flush=True)
        return False
    k = cfg.AGREGAT
    S, SW = agreger(s, k), agreger(sw, k)
    with np.errstate(invalid="ignore", divide="ignore"):
        val = np.where(SW > 0, S / SW, np.nan)
    la_k = la[: (len(la) // k) * k].reshape(-1, k).mean(1)
    lo_k = lo[: (len(lo) // k) * k].reshape(-1, k).mean(1)
    ds = xr.Dataset({"valeur": (("lat", "lon"), val), "poids": (("lat", "lon"), SW),
                     "n_fichiers_valides": (("lat", "lon"), agreger(nj, k))},
                    coords={"lat": la_k, "lon": lo_k},
                    attrs={"unite_source": unite, "n_fichiers": n_fichiers,
                           "n_echecs": sum(r is None for r in resultats)})
    ds.to_netcdf(cfg.DATA_MOIS / f"{mois}.nc")
    return True


def convertir(da):
    """Unités du fichier -> unités d'analyse (config.CONVERSION)."""
    u = da.attrs.get("unite_source", "").lower().replace(" ", "")
    if cfg.CONVERSION == "mm/mois":
        jours = xr.DataArray(pd.DatetimeIndex(da.time.values).days_in_month, dims="time", coords={"time": da.time})
        if "hr" in u or "/h" in u:
            return da * 24 * jours
        if "day" in u or "/d" in u:
            return da * jours
        raise ValueError(f"Unité inconnue : {u!r} – renseigner CONVERSION dans config.py")
    return da * cfg.FACTEUR


# =====================================================================
# Frontières du Maroc (Natural Earth) et moyenne sur le pays
# =====================================================================
NE = "ne_50m_admin_0_countries"
NE_URL = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/50m_cultural/" + NE


def geometrie_zone():
    import geopandas as gpd
    dossier = cfg.DATA_BRUT / "natural_earth"
    dossier.mkdir(parents=True, exist_ok=True)
    for ext in ("shp", "shx", "dbf", "prj"):
        dst = dossier / f"{NE}.{ext}"
        if not dst.exists():
            urlretrieve(f"{NE_URL}.{ext}", dst)
    pays = gpd.read_file(dossier / f"{NE}.shp")
    noms = ["Morocco", "Western Sahara"] if cfg.INCLURE_SAHARA else ["Morocco"]
    return pays[pays.ADMIN.isin(noms)].geometry.union_all()


def masque_zone(lat, lon):
    """True pour les cellules dont le centre est au Maroc."""
    import shapely
    LON, LAT = np.meshgrid(lon, lat)
    m = shapely.contains_xy(geometrie_zone(), LON, LAT)
    if m.sum() == 0:
        raise ValueError("Aucune cellule dans la zone : vérifier ETENDUE / AGREGAT")
    return xr.DataArray(m, dims=("lat", "lon"), coords={"lat": lat, "lon": lon})


def moyenne(x, masque):
    """Moyenne sur les cellules du Maroc, pondérée par cos(latitude)."""
    w = np.cos(np.deg2rad(x.lat)) * masque
    return x.weighted(w.fillna(0)).mean(("lat", "lon"))


# =====================================================================
# Outils statistiques
# =====================================================================
def temps_annees(idx):
    idx = pd.DatetimeIndex(idx)
    return np.asarray(idx.year + (idx.month - 0.5) / 12, dtype=float)


def regression(x, nmin=20):
    """Pente (par décennie) et p-value en chaque point, corrigée de l'autocorrélation lag-1."""
    t = xr.DataArray(temps_annees(x.time.values), coords={"time": x.time}, dims="time")
    m = x.notnull()
    n = m.sum("time")
    tm, ym = t.where(m).mean("time"), x.mean("time")
    sxx = ((t - tm) ** 2).where(m).sum("time")
    pente = ((t - tm) * (x - ym)).sum("time") / sxx
    res = x - (ym + pente * (t - tm))
    r1 = (res * res.shift(time=1)).sum("time") / (res ** 2).sum("time")
    neff = (n * (1 - r1) / (1 + r1)).clip(min=3, max=None)
    neff = neff.where(neff <= n, n)
    se = np.sqrt((res ** 2).sum("time") / (n - 2) / sxx)
    with np.errstate(invalid="ignore", divide="ignore"):
        p = 2 * stats.t.sf(np.abs((pente / se).values), np.maximum(neff.values - 2, 1))
    p = xr.DataArray(p, coords=pente.coords, dims=pente.dims).where(n >= nmin)
    return (pente * 10).where(n >= nmin), p


def analyser_serie(s):
    """s : Series pandas (index = dates). Régression + IC 95 % + Mann-Kendall + pente de Sen."""
    s = s.dropna()
    t = temps_annees(s.index)
    r = stats.linregress(t, s.values)
    res = s.values - (r.intercept + r.slope * t)
    r1 = np.corrcoef(res[1:], res[:-1])[0, 1]
    neff = float(np.clip(len(s) * (1 - r1) / (1 + r1), 3, len(s)))
    se = np.sqrt(np.sum(res ** 2) / (len(s) - 2) / np.sum((t - t.mean()) ** 2))
    tc = stats.t.ppf(0.975, max(neff - 2, 1))
    m1, m2 = mk.original_test(s.values), mk.hamed_rao_modification_test(s.values)
    return dict(pente_dec=r.slope * 10, ic_bas=(r.slope - tc * se) * 10, ic_haut=(r.slope + tc * se) * 10,
                p_reg=float(2 * stats.t.sf(abs(r.slope / se), max(neff - 2, 1))),
                sen_dec=m1.slope * 20,                     # pente de Sen : par semestre -> par décennie
                mk_p=m1.p, mk_hr_p=m2.p, mk_tendance=m1.trend, n=len(s),
                intercept=r.intercept, slope=r.slope)


def significatif(r):
    return bool(r["p_reg"] < cfg.ALPHA and r["mk_hr_p"] < cfg.ALPHA)


# =====================================================================
# Semestres : 1re année = mois disponibles jusqu'à décembre ; ensuite S1 = janv-juin, S2 = juil-déc
# =====================================================================
def stats_sem(x):
    """Anomalie et z-score d'un semestre par rapport aux semestres du même type (S1 / S2)."""
    pa, pz = [], []
    for k in (1, 2):
        sub = x.isel(sem=np.where(x.sem.values % 10 == k)[0])
        a = sub - sub.mean("sem")
        pa.append(a)
        pz.append(a / sub.std("sem", ddof=1))
    return xr.concat(pa, "sem").sortby("sem"), xr.concat(pz, "sem").sortby("sem")


def temps_semestres(codes):
    """Date au milieu de chaque semestre (1er avril ou 1er octobre)."""
    return pd.DatetimeIndex([pd.Timestamp(int(c) // 10, 4 if int(c) % 10 == 1 else 10, 1) for c in codes])


def construire_semestres(da):
    """Dataset semestriel : moyennes, anomalies, z-scores, tendances (1 ou 2 périodes)."""
    sem_np = da.time.dt.year.values * 10 + np.where(da.time.dt.month.values <= 6, 1, 2)
    nb = pd.Series(sem_np).value_counts()
    garder = np.isin(sem_np, nb[nb >= cfg.MOIS_MIN_SEMESTRE].index)    # semestres trop courts écartés
    da, sem_np = da.isel(time=np.where(garder)[0]), sem_np[garder]
    grp = xr.DataArray(sem_np, dims="time", coords={"time": da.time}, name="sem")
    da_sem = da.groupby(grp).mean("time")
    nb = pd.Series(sem_np).value_counts().sort_index()
    nb_mois = xr.DataArray(nb.values, dims="sem", coords={"sem": nb.index.values})
    anom_sem, z_sem = stats_sem(da_sem)
    anom_t = anom_sem.rename(sem="time").assign_coords(time=temps_semestres(da_sem.sem.values))
    pente, pval = regression(anom_t)
    champs = {"val_sem": da_sem, "anom_sem": anom_sem, "z_sem": z_sem, "nb_mois": nb_mois,
              "val_moyen": da.mean("time"), "pente": pente, "pval": pval}
    if cfg.FIN_PRE:
        champs["pente_pre"], champs["pval_pre"] = regression(anom_t.sel(time=slice(None, cfg.FIN_PRE)))
    return xr.Dataset(champs)


def charger_contexte():
    """Relit les résultats de l'étape 4 et prépare tout ce qu'il faut pour les graphiques."""
    ds = xr.open_dataset(cfg.DATA_TRAITE / "semestres.nc").load()
    mensuel = xr.open_dataarray(cfg.DATA_TRAITE / "mensuel.nc").load()
    masque = xr.open_dataarray(cfg.DATA_TRAITE / "masque.nc").load().astype(bool)
    codes = ds.sem.values.astype(int)
    etiq = [f"{c // 10}-S{c % 10}" for c in codes]
    t_sem = temps_semestres(codes)
    m = moyenne(ds["val_sem"], masque)
    a, z = stats_sem(m)
    evt = None
    if cfg.FIN_PRE:
        apres = np.where(t_sem > pd.Timestamp(cfg.FIN_PRE))[0]
        evt = int(apres[0]) if len(apres) else None
    return SimpleNamespace(
        zone=cfg.ZONE, ds=ds, masque=masque, codes=codes, etiq=etiq, t_sem=t_sem, n_sem=len(codes),
        m=m, a=a, z=z,
        serie_sem=pd.Series(m.values, index=t_sem),
        serie_anom=pd.Series(a.values, index=t_sem),
        serie_mois=moyenne(mensuel, masque).to_series(),
        nb_mois=pd.Series(ds["nb_mois"].values, index=codes),
        evenement=evt,
    )


def periodes():
    """[(libellé, date de fin)] : période complète, puis période avant la coupure (si FIN_PRE)."""
    return [("complete", None)] + ([("pre", cfg.FIN_PRE)] if cfg.FIN_PRE else [])


def zoom(x):
    e = cfg.ETENDUE
    return x.sel(lat=slice(e[2], e[3]), lon=slice(e[0], e[1]))


def texte_zone():
    return "le Maroc" + (" (avec les provinces du Sud)" if cfg.INCLURE_SAHARA else " (sans les provinces du Sud)")


def vers_df(champs):
    """Dict {nom: DataArray} -> DataFrame (une ligne par point de grille)."""
    df = xr.Dataset(champs).reset_coords(drop=True).to_dataframe().reset_index()
    return df.dropna(subset=[list(champs)[0]])


In [ ]:
%%writefile traces.py
"""Petites fonctions de tracé communes aux graphiques."""
import numpy as np

import config as cfg


def axe_semestres(ax, ctx):
    x = np.arange(ctx.n_sem)
    ax.set_xticks(x)
    ax.set_xticklabels(ctx.etiq, rotation=90, fontsize=7)
    ax.set_xlim(-.5, ctx.n_sem - .5)


def ligne_evenement(ax, ctx):
    """Repère la coupure entre les deux périodes (config.FIN_PRE / NOM_COUPURE)."""
    if ctx.evenement is not None:
        ax.axvline(ctx.evenement - .5, color="darkorange", ls="--", lw=1.3)
        ax.text(ctx.evenement - .5, .98, " " + cfg.NOM_COUPURE, color="darkorange", fontsize=8,
                va="top", transform=ax.get_xaxis_transform())


def frontiere(ax, geom, ccrs):
    ax.add_geometries([geom], crs=ccrs.PlateCarree(), facecolor="none", edgecolor="black", linewidth=1.3)


## Étape 1 – Recherche des fichiers Earthdata
Compte gratuit : https://urs.earthdata.nasa.gov. Les identifiants sont **saisis au clavier quand la cellule les demande** : ne les écrivez jamais dans le notebook.

Option plus pratique : les **Secrets** de Colab (icône clé à gauche), nommés `EARTHDATA_USERNAME` et `EARTHDATA_PASSWORD`. Dans ce cas, exécutez d'abord :
```python
import os
from google.colab import userdata
os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")
```

In [ ]:
"""ÉTAPE 1 – Recherche des fichiers NASA (Earthdata) et index.

Aucun gros téléchargement ici : l'étape 3 ne lit, dans chaque fichier, que la fenêtre du Maroc.
Prérequis : un compte gratuit https://urs.earthdata.nasa.gov
Produit : data/brut/index.csv (url, date, mois).
"""
import earthaccess
import pandas as pd

import config as cfg

EXT = (".nc", ".nc4", ".he5", ".h5", ".hdf5")


def main():
    earthaccess.login(persist=True)
    resultats = earthaccess.search_data(short_name=cfg.PRODUIT, version=cfg.VERSION,
                                        temporal=(cfg.DATE_DEBUT, cfg.DATE_FIN))
    print(len(resultats), "fichiers trouvés")
    lignes = []
    for r in resultats:
        liens = r.data_links()
        liens = [l for l in liens if l.lower().endswith(EXT)] or liens
        debut = pd.Timestamp(r["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"])
        debut = debut.tz_convert(None) if debut.tzinfo else debut
        lignes.append({"url": liens[0], "date": debut.normalize(), "mois": debut.strftime("%Y-%m")})
    index = pd.DataFrame(lignes).sort_values("date").drop_duplicates("date", keep="last")
    index.to_csv(cfg.DATA_BRUT / "index.csv", index=False)
    par_mois = index.groupby("mois").size()
    print(f"Index enregistré : {cfg.DATA_BRUT / 'index.csv'}")
    print(f"{len(index)} fichiers, {len(par_mois)} mois : de {par_mois.index[0]} à {par_mois.index[-1]}"
          f" ({par_mois.min()} à {par_mois.max()} fichier(s) par mois)")


main()

## Étape 2 – Inspection de la structure d'un fichier
Affiche les tableaux d'un fichier, la variable choisie et un aperçu des valeurs sur le Maroc. Si le choix automatique ne convient pas, renseignez `VARIABLE` dans `config.py`.

In [ ]:
"""ÉTAPE 2 – Inspection de la structure d'un fichier.

Affiche les tableaux du premier fichier, puis la variable, le poids et l'indicateur de qualité
qui seront utilisés. Si le choix automatique ne convient pas, renseignez VARIABLE (et MOTS_POIDS,
MOTS_QUALITE, QUALITE_OK) dans config.py.
"""
import os

import h5py
import numpy as np
import pandas as pd

import config as cfg
import utils


def main():
    index = pd.read_csv(cfg.DATA_BRUT / "index.csv")
    url = index.url.iloc[len(index) // 2]          # un fichier au milieu de la période
    print("Fichier :", url.split("/")[-1], "\n")
    local = os.path.exists(url)
    if not local:
        utils.preparer_acces()
    with (open(url, "rb") if local else utils._fs().open(url, "rb")) as fo, h5py.File(fo, "r") as f:
        for p, d in utils.lister(f).items():
            print(f"  {p:45s} {str(d.shape):18s} {utils._attr(d, 'units') or ''}")
        var = cfg.VARIABLE or utils.trouver(f, cfg.MOTS_VARIABLE, cfg.EXCLURE)
        poids = utils.trouver(f, cfg.MOTS_POIDS)
        qualite = utils.trouver(f, cfg.MOTS_QUALITE)
        print(f"\n>>> Variable : {var} | Poids : {poids} | Qualité : {qualite}")
        if qualite:
            q = utils.lire_fenetre(f, qualite, cfg.ETENDUE)[0]
            v, n = np.unique(q[np.isfinite(q)], return_counts=True)
            print("    valeurs de l'indicateur de qualité (fenêtre) :", dict(zip(v.tolist(), n.tolist())),
                  "| conservées :", cfg.QUALITE_OK)
    a, w, la, lo, unite = utils.lire_url(url)
    ok = np.isfinite(a)
    print(f"\nFenêtre lue : {a.shape[0]} lat x {a.shape[1]} lon "
          f"({la.min():.2f} à {la.max():.2f} N, {lo.min():.2f} à {lo.max():.2f} E)")
    print(f"Unité du fichier : {unite!r} | valeurs valides : {ok.mean() * 100:.1f} %")
    if ok.any():
        print(f"Min / médiane / max : {np.nanmin(a):.4g} / {np.nanmedian(a):.4g} / {np.nanmax(a):.4g}")
    else:
        print("ATTENTION : aucune valeur valide – vérifier VARIABLE et QUALITE_OK (None = pas de filtre) dans config.py")


main()

## Étape 3 – Extraction, moyennes mensuelles, unités
Environ 8 000 fichiers journaliers : **1 à 3 heures**, reprenable à tout moment. Seule la fenêtre `ETENDUE` est lue dans chaque fichier (pas de téléchargement complet). La ligne « Mois lus » doit afficher environ 265 mois.

In [ ]:
"""ÉTAPE 3 – Extraction (fenêtre du Maroc), moyennes mensuelles, nettoyage, unités.

Les fichiers sont lus par plusieurs processus en parallèle ; seule la fenêtre ETENDUE est lue.
Chaque mois est enregistré dans data/mois/AAAA-MM.nc dès qu'il est fini : si Colab se déconnecte,
relancez la préparation puis cette cellule, elle reprend où elle s'était arrêtée.
Produit : data/traite/mensuel.nc (temps, lat, lon) et data/traite/masque.nc (cellules du Maroc).
"""
import os
import time

import numpy as np
import pandas as pd
import xarray as xr

import config as cfg
import utils


def a_refaire(mois, n):
    p = cfg.DATA_MOIS / f"{mois}.nc"
    if not p.exists():
        return True
    with xr.open_dataset(p) as ds:
        return int(ds.attrs.get("n_fichiers", 0)) < n      # mois incomplet lors d'un passage précédent


def lots(mois, groupes, taille):
    """Regroupe des mois consécutifs pour avoir au moins `taille` fichiers à lire en parallèle."""
    lot = []
    for m in mois:
        lot.append(m)
        if sum(len(groupes[x]) for x in lot) >= taille:
            yield lot
            lot = []
    if lot:
        yield lot


def extraire():
    index = pd.read_csv(cfg.DATA_BRUT / "index.csv")
    groupes = index.groupby("mois").url.apply(list)
    a_faire = [m for m in groupes.index if a_refaire(m, len(groupes[m]))]
    print(f"{len(groupes)} mois au total, {len(groupes) - len(a_faire)} déjà extraits, {len(a_faire)} à extraire",
          flush=True)
    if not a_faire:
        return
    if not os.path.exists(index.url.iloc[0]):
        utils.preparer_acces()
    t0, faits = time.time(), 0
    with utils.processus() as ex:
        for lot in lots(a_faire, groupes, 4 * cfg.PARALLELE):
            urls = [u for m in lot for u in groupes[m]]
            res = dict(zip(urls, ex.map(utils.lire_url, urls)))
            for m in lot:
                utils.enregistrer_mois(m, [res[u] for u in groupes[m]], len(groupes[m]))
            faits += len(lot)
            dt = time.time() - t0
            print(f"  {lot[-1]} fini ({faits}/{len(a_faire)} mois) – écoulé {dt / 60:.1f} min, "
                  f"reste ~{dt / faits * (len(a_faire) - faits) / 60:.0f} min", flush=True)


def assembler():
    fichiers = sorted(cfg.DATA_MOIS.glob("*.nc"))
    morceaux, echecs, unite = [], {}, ""
    for p in fichiers:
        with xr.open_dataset(p) as ds:
            morceaux.append(ds.valeur.load().expand_dims(time=[pd.Timestamp(p.stem + "-01")]))
            unite = ds.attrs.get("unite_source", unite)
            if int(ds.attrs.get("n_echecs", 0)):
                echecs[p.stem] = int(ds.attrs["n_echecs"])
    da = xr.concat(morceaux, "time", join="override").sortby("time")
    da = da.sel(time=slice(cfg.DATE_DEBUT, cfg.DATE_FIN))
    da.attrs["unite_source"] = unite
    da = utils.convertir(da)
    da.name, da.attrs = cfg.NOM, {"unite": cfg.UNITE}
    assert da.sizes["time"] > 24, "Trop peu de mois : vérifier l'étape 1"

    print("Mois lus :", da.sizes["time"], "| de", str(da.time.values[0])[:7], "à", str(da.time.values[-1])[:7])
    print(f"Grille : {da.sizes['lat']} x {da.sizes['lon']} cellules de {float(np.diff(da.lon)[0]):.2f}°")
    print("Valeurs manquantes (%) :", round(float(da.isnull().mean()) * 100, 1))
    print("Médiane :", round(float(da.median()), 3), cfg.UNITE)
    if echecs:
        print("Fichiers illisibles (par mois) :", echecs)
    da.to_netcdf(cfg.DATA_TRAITE / "mensuel.nc")

    masque = utils.masque_zone(da.lat.values, da.lon.values)
    masque.astype("int8").rename("masque").to_netcdf(cfg.DATA_TRAITE / "masque.nc")
    print(f"Cellules dans {utils.texte_zone()} : {int(masque.sum())}")
    print("Enregistré :", cfg.DATA_TRAITE / "mensuel.nc")


extraire()
assembler()

## Étape 4 – Semestres, anomalies, tendances

In [ ]:
"""ÉTAPE 4 – Passage aux semestres, anomalies, tendances.

Découpage : 1re année = mois disponibles jusqu'à décembre (S2), puis S1 = janvier -> juin
et S2 = juillet -> décembre. Anomalie / z-score = écart aux semestres du même type.
Tendances sur toute la période (et avant FIN_PRE si renseigné dans config.py).

Produit : data/traite/semestres.nc
"""
import xarray as xr

import config as cfg
import utils


def main():
    da = xr.open_dataarray(cfg.DATA_TRAITE / "mensuel.nc").load()
    ds = utils.construire_semestres(da)
    ds.to_netcdf(cfg.DATA_TRAITE / "semestres.nc")
    tous = sorted(set(da.time.dt.year.values * 10 + (da.time.dt.month.values > 6) + 1))
    ecartes = [f"{c // 10}-S{c % 10}" for c in tous if c not in ds.sem.values]
    if ecartes:
        print(f"Semestres écartés (moins de {cfg.MOIS_MIN_SEMESTRE} mois) :", ecartes)

    ctx = utils.charger_contexte()
    print("Zone :", utils.texte_zone())
    print(ctx.n_sem, "semestres :", ctx.etiq[0], "->", ctx.etiq[-1])
    incomplets = [f"{e} ({ctx.nb_mois[c]} mois)" for e, c in zip(ctx.etiq, ctx.codes) if ctx.nb_mois[c] < 6]
    if incomplets:
        print("Semestres incomplets :", incomplets)
    print(f"Moyenne sur le Maroc : {ctx.serie_sem.mean():.3f} {cfg.UNITE}")
    print("Enregistré dans", cfg.DATA_TRAITE)


main()

## Étape 5 – Graphique 1 : cartes du Maroc
À gauche la moyenne, à droite la tendance par décennie (hachures = significatif). Trait noir = frontières utilisées pour la moyenne.

In [ ]:
"""ÉTAPE 5 – Graphique 1 : cartes du Maroc – moyenne et tendance (hachures = significatif)."""
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

import config as cfg
import traces
import utils


def main():
    ctx = utils.charger_contexte()
    ds, pc = ctx.ds, ccrs.PlateCarree()
    geom = utils.geometrie_zone()
    moy, pn = utils.zoom(ds.val_moyen), utils.zoom(ds.pente)
    sig = (utils.zoom(ds.pval) < cfg.ALPHA).astype(float)
    lim = float(np.nanpercentile(np.abs(pn), 98))
    lim = lim if np.isfinite(lim) and lim > 0 else 1.0
    r = utils.analyser_serie(ctx.serie_anom)
    fin = ctx.etiq[-1][:4]

    fig, axes = plt.subplots(1, 2, figsize=(15, 7.2), subplot_kw={"projection": pc})
    im0 = axes[0].pcolormesh(moy.lon, moy.lat, moy, cmap=cfg.CMAP_MOYENNE, shading="auto", transform=pc)
    im1 = axes[1].pcolormesh(pn.lon, pn.lat, pn, cmap=cfg.CMAP_TENDANCE, vmin=-lim, vmax=lim,
                             shading="auto", transform=pc)
    if sig.sum() > 0:
        axes[1].contourf(sig.lon, sig.lat, sig, levels=[.5, 1.5], colors="none", hatches=["...."], transform=pc)
    for ax in axes:
        ax.set_extent(cfg.ETENDUE, crs=pc)
        ax.coastlines(lw=.6)
        ax.add_feature(cfeature.BORDERS, lw=.3, ls=":")
        traces.frontiere(ax, geom, ccrs)
        gl = ax.gridlines(draw_labels=True, lw=.2)
        gl.top_labels = gl.right_labels = False
    plt.colorbar(im0, ax=axes[0], orientation="horizontal", pad=.07, label=cfg.UNITE)
    plt.colorbar(im1, ax=axes[1], orientation="horizontal", pad=.07, label=f"{cfg.UNITE} par décennie")
    axes[1].legend(handles=[Patch(facecolor="none", hatch="....", label=f"significatif (p < {cfg.ALPHA})")],
                   loc="lower left", fontsize=8)
    axes[0].set_title(f"Moyenne {ctx.etiq[0][:4]}–{fin}\nMaroc : {ctx.serie_sem.mean():.3g} {cfg.UNITE}", fontsize=11)
    axes[1].set_title(f"Tendance par décennie (semestres)\nMaroc : {r['pente_dec']:+.3g} {cfg.UNITE}/décennie, "
                      f"p = {r['p_reg']:.3f}", fontsize=11)
    fig.suptitle(f"1. Carte – {cfg.TITRE} sur {utils.texte_zone()}", fontsize=13)
    fig.savefig(cfg.SORTIE_ZONE / "1_carte.png", dpi=170, bbox_inches="tight")
    if cfg.AFFICHER:
        plt.show()


main()

## Étape 6 – Graphique 2 : série temporelle (un point tous les 6 mois)

In [ ]:
"""ÉTAPE 6 – Graphique 2 : série temporelle du Maroc (un point tous les 6 mois) et tendance."""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config as cfg
import traces
import utils


def main():
    ctx = utils.charger_contexte()
    y = ctx.serie_anom
    r = utils.analyser_serie(y)
    x = np.arange(ctx.n_sem)
    t = utils.temps_annees(ctx.t_sem)
    ybar, tbar = y.mean(), t.mean()
    fin = ctx.etiq[-1][:4]

    fig, ax = plt.subplots(figsize=(14, 5.5))
    ax.fill_between(x, ybar + r["ic_bas"] / 10 * (t - tbar), ybar + r["ic_haut"] / 10 * (t - tbar),
                    color="#9ec5ee", alpha=.5, label=f"IC 95 % ({ctx.etiq[0][:4]}–{fin})")
    ax.bar(x, y.values, color=np.where(y.values >= 0, "#5b9bd5", "#d9895b"), alpha=.55, width=.8)
    ax.plot(x, y.values, color="grey", lw=1, marker="o", ms=4, label="Anomalie semestrielle")
    ax.plot(x, r["intercept"] + r["slope"] * t, color="navy", lw=2, label=f"Tendance {ctx.etiq[0][:4]}–{fin}")
    texte = (f"{ctx.etiq[0][:4]}–{fin} : {r['pente_dec']:+.3g} [{r['ic_bas']:+.3g} ; {r['ic_haut']:+.3g}] "
             f"{cfg.UNITE}/décennie, p = {r['p_reg']:.3f}, MK corrigé p = {r['mk_hr_p']:.3f}, Sen {r['sen_dec']:+.3g}")
    if cfg.FIN_PRE:
        r2 = utils.analyser_serie(y[y.index <= cfg.FIN_PRE])
        npre = int((ctx.t_sem <= pd.Timestamp(cfg.FIN_PRE)).sum())
        ax.plot(x[:npre], r2["intercept"] + r2["slope"] * t[:npre], color="darkorange", lw=2, ls="--",
                label=f"Tendance {ctx.etiq[0][:4]}–{cfg.FIN_PRE[:4]}")
        texte += (f"\n{ctx.etiq[0][:4]}–{cfg.FIN_PRE[:4]} : {r2['pente_dec']:+.3g} [{r2['ic_bas']:+.3g} ; "
                  f"{r2['ic_haut']:+.3g}] {cfg.UNITE}/décennie, p = {r2['p_reg']:.3f}, MK corrigé p = {r2['mk_hr_p']:.3f}")
    traces.ligne_evenement(ax, ctx)
    ax.axhline(0, color="k", lw=.5)
    ax.grid(alpha=.3)
    traces.axe_semestres(ax, ctx)
    ax.set_ylabel(f"Anomalie ({cfg.UNITE})")
    ax.legend(fontsize=8, loc="upper left")
    ax.text(.99, .03, texte, transform=ax.transAxes, ha="right", va="bottom", fontsize=8,
            bbox=dict(fc="white", alpha=.85))
    ax.set_title(f"2. Graphe temporel – {cfg.TITRE}, Maroc, {ctx.etiq[0][:4]} à {fin}, un point tous les 6 mois",
                 fontsize=12)
    fig.savefig(cfg.SORTIE_ZONE / "2_graphe_temporel.png", dpi=170, bbox_inches="tight")
    if cfg.AFFICHER:
        plt.show()


main()

## Étape 7 – Graphique 3 : cycle saisonnier
Remplace le profil vertical du notebook H₂O (cette variable n'a pas de niveaux de pression) : moyenne de chaque mois sur le Maroc, chaque année en gris.

In [ ]:
"""ÉTAPE 7 – Graphique 3 : cycle saisonnier (moyenne de chaque mois sur le Maroc)."""
import matplotlib.pyplot as plt
import pandas as pd

import config as cfg
import utils

MOIS = ["janv.", "févr.", "mars", "avr.", "mai", "juin", "juil.", "août", "sept.", "oct.", "nov.", "déc."]


def tableau_cycle(ctx):
    s = ctx.serie_mois
    df = pd.DataFrame({"val": s.values, "annee": s.index.year, "mois": s.index.month})
    tab = df.groupby("mois").val.agg(moyenne="mean", ecart_type="std", minimum="min", maximum="max")
    if cfg.FIN_PRE:
        coupure = pd.Timestamp(cfg.FIN_PRE)
        tab[f"moyenne_jusqu_{cfg.FIN_PRE[:4]}"] = df[s.index <= coupure].groupby("mois").val.mean()
        tab[f"moyenne_apres_{cfg.FIN_PRE[:4]}"] = df[s.index > coupure].groupby("mois").val.mean()
    return df, tab.reset_index()


def main():
    ctx = utils.charger_contexte()
    df, tab = tableau_cycle(ctx)
    fin = ctx.etiq[-1][:4]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.axvspan(.5, 6.5, color="#f2f2f2", zorder=0)
    ax.text(3.5, .98, "S1", transform=ax.get_xaxis_transform(), ha="center", va="top", color="grey")
    ax.text(9.5, .98, "S2", transform=ax.get_xaxis_transform(), ha="center", va="top", color="grey")
    for _, g in df.groupby("annee"):
        ax.plot(g.mois, g.val, color="grey", lw=.6, alpha=.3)
    ax.fill_between(tab.mois, tab.moyenne - tab.ecart_type, tab.moyenne + tab.ecart_type,
                    color="#9ec5ee", alpha=.5, label="± 1 écart-type")
    ax.plot(tab.mois, tab.moyenne, color="navy", lw=2.5, marker="o", label=f"Moyenne {ctx.etiq[0][:4]}–{fin}")
    if cfg.FIN_PRE:
        ax.plot(tab.mois, tab.iloc[:, -2], color="darkorange", lw=1.8, ls="--", label=f"Moyenne jusqu'à {cfg.FIN_PRE[:4]}")
        ax.plot(tab.mois, tab.iloc[:, -1], color="crimson", lw=1.8, ls="--", label=f"Moyenne après {cfg.FIN_PRE[:4]}")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MOIS)
    ax.set_xlim(.5, 12.5)
    ax.grid(alpha=.3)
    ax.set_ylabel(cfg.UNITE)
    ax.legend(fontsize=8, loc="best")
    ax.set_title(f"3. Cycle saisonnier – {cfg.TITRE}, Maroc (lignes grises = chaque année)", fontsize=12)
    fig.savefig(cfg.SORTIE_ZONE / "3_cycle_saisonnier.png", dpi=170, bbox_inches="tight")
    if cfg.AFFICHER:
        plt.show()


main()

## Étape 8 – Graphique 4 : détection d'anomalies
Semestres exceptionnellement hauts en rouge, exceptionnellement bas en bleu.

In [ ]:
"""ÉTAPE 8 – Graphique 4 : détection d'anomalies (semestres exceptionnellement hauts ou bas)."""
import matplotlib.pyplot as plt
import numpy as np

import config as cfg
import traces
import utils


def main():
    ctx = utils.charger_contexte()
    an, zn = ctx.a.values, ctx.z.values
    sd = {k: float(np.nanstd(an[ctx.codes % 10 == k], ddof=1)) for k in (1, 2)}
    bande = np.array([cfg.Z_SEUIL * sd[c % 10] for c in ctx.codes])
    haut, bas = zn >= cfg.Z_SEUIL, zn <= -cfg.Z_SEUIL
    x = np.arange(ctx.n_sem)

    fig, ax = plt.subplots(figsize=(14, 5.5))
    ax.fill_between(x, -bande, bande, color="grey", alpha=.25, label=f"Normal (±{cfg.Z_SEUIL} σ)")
    ax.axhline(0, color="k", lw=.6)
    ax.plot(x, an, color="#2b3a4a", lw=1.5, marker="o", ms=3)
    for sel, coul, lib, dy in [(haut, "red", "exceptionnellement haut", 8), (bas, "#1f5fbf", "exceptionnellement bas", -14)]:
        ax.scatter(x[sel], an[sel], color=coul, s=60, zorder=5, label=f"Semestre {lib}")
        for xi, yi, e in zip(x[sel], an[sel], np.array(ctx.etiq)[sel]):
            ax.annotate(e, (xi, yi), textcoords="offset points", xytext=(0, dy), ha="center", fontsize=8, color=coul)
    traces.ligne_evenement(ax, ctx)
    traces.axe_semestres(ax, ctx)
    ax.margins(y=.15)
    ax.grid(alpha=.3)
    ax.legend(fontsize=8, loc="upper left")
    ax.set_ylabel(f"Anomalie ({cfg.UNITE})")
    ax.set_title(f"4. Détection d'anomalies – {cfg.TITRE}, Maroc (semestres exceptionnels)", fontsize=12)
    fig.savefig(cfg.SORTIE_ZONE / "4_detection_anomalies.png", dpi=170, bbox_inches="tight")
    print("Semestres exceptionnellement hauts :", [str(e) for e in np.array(ctx.etiq)[haut]] or "aucun")
    print("Semestres exceptionnellement bas   :", [str(e) for e in np.array(ctx.etiq)[bas]] or "aucun")
    if cfg.AFFICHER:
        plt.show()


main()

## Étape 9 – Export CSV
Un CSV par graphique dans `resultats/hcho/`.

In [ ]:
"""ÉTAPE 9 – Export des résultats en CSV (un fichier par graphique) + archive ZIP.

Produit dans resultats/<variable>/ : 1_carte.csv, 2_graphe_temporel.csv, 3_cycle_saisonnier.csv,
4_detection_anomalies.csv, mensuel_maroc.csv, resultats_<variable>_maroc.zip
"""
import zipfile

import numpy as np
import pandas as pd

import config as cfg
import utils


def main():
    ctx = utils.charger_contexte()
    ds, sortie = ctx.ds, cfg.SORTIE_ZONE
    enc = dict(index=False, encoding="utf-8-sig")
    u = cfg.UNITE

    # 1. Carte
    champs = {f"moyenne ({u})": ds.val_moyen, f"tendance ({u}/décennie)": ds.pente, "p_value": ds.pval}
    if cfg.FIN_PRE:
        champs[f"tendance_jusqu_{cfg.FIN_PRE[:4]} ({u}/décennie)"] = ds.pente_pre
        champs[f"p_value_jusqu_{cfg.FIN_PRE[:4]}"] = ds.pval_pre
    df1 = utils.vers_df({k: utils.zoom(v) for k, v in champs.items()})
    df1["significatif"] = df1.p_value < cfg.ALPHA
    m = ctx.masque.to_dataframe("au_maroc").reset_index()
    df1 = df1.merge(m, on=["lat", "lon"], how="left")
    df1.round(5).to_csv(sortie / "1_carte.csv", **enc)

    # 2. Graphe temporel
    y = ctx.serie_anom
    r = utils.analyser_serie(y)
    t = utils.temps_annees(ctx.t_sem)
    df2 = pd.DataFrame(dict(semestre=ctx.etiq, date_milieu=ctx.t_sem.strftime("%Y-%m-%d"),
                            nb_mois=ctx.nb_mois.values, moyenne=ctx.serie_sem.values, anomalie=y.values,
                            ligne_tendance=r["intercept"] + r["slope"] * t))
    for k in ("pente_dec", "ic_bas", "ic_haut", "p_reg", "mk_p", "mk_hr_p", "sen_dec", "mk_tendance"):
        df2[k] = r[k]
    if cfg.FIN_PRE:
        r2 = utils.analyser_serie(y[y.index <= cfg.FIN_PRE])
        for k in ("pente_dec", "ic_bas", "ic_haut", "p_reg", "mk_hr_p"):
            df2[f"{k}_jusqu_{cfg.FIN_PRE[:4]}"] = r2[k]
    df2.insert(3, "unite", u)
    df2.round(5).to_csv(sortie / "2_graphe_temporel.csv", **enc)

    # 3. Cycle saisonnier + série mensuelle
    s = ctx.serie_mois
    df = pd.DataFrame({"val": s.values, "mois": s.index.month})
    tab = df.groupby("mois").val.agg(moyenne="mean", ecart_type="std", minimum="min", maximum="max").reset_index()
    tab.round(5).to_csv(sortie / "3_cycle_saisonnier.csv", **enc)
    pd.DataFrame({"mois": s.index.strftime("%Y-%m"), f"moyenne_maroc ({u})": s.values}).round(5) \
        .to_csv(sortie / "mensuel_maroc.csv", **enc)

    # 4. Détection d'anomalies
    df4 = pd.DataFrame(dict(semestre=ctx.etiq, nb_mois=ctx.nb_mois.values, moyenne=ctx.m.values,
                            anomalie=ctx.a.values, z_score=ctx.z.values))
    df4["exceptionnellement_haut"] = df4.z_score >= cfg.Z_SEUIL
    df4["exceptionnellement_bas"] = df4.z_score <= -cfg.Z_SEUIL
    df4.round(4).to_csv(sortie / "4_detection_anomalies.csv", **enc)

    with zipfile.ZipFile(sortie / f"resultats_{cfg.NOM}_maroc.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(sortie.iterdir()):
            if f.suffix in (".csv", ".png"):
                zf.write(f, f.name)
    print("Fichiers écrits dans", sortie)


main()

## Étape 10 – Synthèse : quoi / où / combien / significatif

In [ ]:
"""ÉTAPE 10 – Synthèse : Quoi ? Où ? Combien ? Est-ce significatif ?"""
import numpy as np

import config as cfg
import utils


def main():
    ctx = utils.charger_contexte()
    y = ctx.serie_anom
    fin = ctx.etiq[-1][:4]
    resultats = [(f"{ctx.etiq[0][:4]}–{fin}", utils.analyser_serie(y))]
    if cfg.FIN_PRE:
        resultats.append((f"{ctx.etiq[0][:4]}–{cfg.FIN_PRE[:4]}", utils.analyser_serie(y[y.index <= cfg.FIN_PRE])))
    moy = ctx.serie_sem.mean()
    pente = utils.zoom(ctx.ds.pente).where(ctx.masque)
    sig = (utils.zoom(ctx.ds.pval) < cfg.ALPHA).where(ctx.masque)
    print(f"QUOI  : {cfg.TITRE} ({cfg.PRODUIT} v{cfg.VERSION}), anomalies semestrielles – {utils.texte_zone()}")
    print(f"        moyenne {ctx.etiq[0][:4]}–{fin} : {moy:.3g} {cfg.UNITE}")
    print(f"OÙ    : {float(sig.mean()) * 100:.0f} % des cellules du Maroc ont une tendance significative ; "
          f"tendance locale de {float(pente.min()):+.3g} à {float(pente.max()):+.3g} {cfg.UNITE}/décennie")
    print("COMBIEN / SIGNIFICATIF :")
    for lib, r in resultats:
        sens = "augmente" if r["pente_dec"] > 0 else "diminue"
        sig_txt = "SIGNIFICATIVE" if utils.significatif(r) else "non significative"
        print(f"  - {lib} : {sens} de {r['pente_dec']:+.3g} {cfg.UNITE}/décennie "
              f"({r['pente_dec'] / moy * 100:+.1f} %/décennie, IC 95 % [{r['ic_bas']:+.3g} ; {r['ic_haut']:+.3g}]) – {sig_txt} "
              f"(p = {r['p_reg']:.3f}, Mann-Kendall corrigé p = {r['mk_hr_p']:.3f}, pente de Sen {r['sen_dec']:+.3g}).")
    haut = [e for e, z in zip(ctx.etiq, ctx.z.values) if z >= cfg.Z_SEUIL]
    bas = [e for e, z in zip(ctx.etiq, ctx.z.values) if z <= -cfg.Z_SEUIL]
    print(f"EXCEPTIONNELS : hauts {haut or 'aucun'} | bas {bas or 'aucun'}")


main()

## Étape 11 – Données du site web
Crée `site/data/donnees_hcho.js` (même structure que le `donnees.js` du H₂O, avec une clé `cycle` en plus et `profils` vide).

In [ ]:
"""ÉTAPE 11 – Prépare les données du site web (aucune authentification NASA n'est nécessaire sur le site).

Même structure que donnees.js du H2O : une seule entrée dans « niveaux » (pas de niveaux de pression
pour cette variable), « profils » vide, et une clé « cycle » en plus (cycle saisonnier).
Produit : site/data/donnees_<variable>.js et site/data/csv/<variable>_*.csv
"""
import json
import shutil
from datetime import date

import numpy as np
import pandas as pd

import config as cfg
import utils


def liste(a, n=4):
    a = np.asarray(a, dtype=float).ravel()
    return [round(float(v), n) if np.isfinite(v) else None for v in a]


def stat(r):
    return dict(pente=float(r["pente_dec"]), ic_bas=float(r["ic_bas"]), ic_haut=float(r["ic_haut"]),
                p_reg=float(r["p_reg"]), mk_p=float(r["mk_p"]), mk_hr_p=float(r["mk_hr_p"]),
                sen=float(r["sen_dec"]), tendance=str(r["mk_tendance"]), n=int(r["n"]))


def main():
    ctx = utils.charger_contexte()
    ds = ctx.ds
    t = utils.temps_annees(ctx.t_sem)
    a_s = ctx.serie_anom
    r = utils.analyser_serie(a_s)
    sd = {k: float(np.nanstd(a_s.values[ctx.codes % 10 == k], ddof=1)) for k in (1, 2)}
    ybar, tbar = float(a_s.mean()), float(t.mean())
    npre, r2 = None, None
    if cfg.FIN_PRE:
        npre = int((ctx.t_sem <= pd.Timestamp(cfg.FIN_PRE)).sum())
        r2 = utils.analyser_serie(a_s[a_s.index <= cfg.FIN_PRE])
    carte = dict(moyenne=liste(utils.zoom(ds.val_moyen).values), pente=liste(utils.zoom(ds.pente).values),
                 sig=[int(v) for v in (utils.zoom(ds.pval).values < cfg.ALPHA).ravel()],
                 au_maroc=[int(v) for v in ctx.masque.values.ravel()])
    if cfg.FIN_PRE:
        carte.update(pente_pre=liste(utils.zoom(ds.pente_pre).values),
                     sig_pre=[int(v) for v in (utils.zoom(ds.pval_pre).values < cfg.ALPHA).ravel()])
    niveau = dict(
        plev=None, moyenne=liste(ctx.m.values), anomalie=liste(a_s.values), z=liste(ctx.z.values, 3),
        bande=liste([cfg.Z_SEUIL * sd[int(c) % 10] for c in ctx.codes]),
        tendance=stat(r), tendance_pre=stat(r2) if r2 else None,
        ligne=liste(r["intercept"] + r["slope"] * t),
        ligne_pre=liste(r2["intercept"] + r2["slope"] * t[:npre]) if r2 else None,
        ic_bas_ligne=liste(ybar + r["ic_bas"] / 10 * (t - tbar)),
        ic_haut_ligne=liste(ybar + r["ic_haut"] / 10 * (t - tbar)),
        carte=carte)
    s = ctx.serie_mois
    clim = pd.Series(s.values, index=s.index.month).groupby(level=0).agg(["mean", "std"])
    lat, lon = utils.zoom(ds.val_moyen).lat.values, utils.zoom(ds.val_moyen).lon.values
    dlon = float(np.median(np.diff(lon)))
    donnees = dict(
        meta=dict(zone=cfg.ZONE, zone_texte=utils.texte_zone(), variable=cfg.NOM, titre=cfg.TITRE,
                  etendue=list(cfg.ETENDUE), boite=None, taille_marqueur=int(min(120, max(4, 40 * dlon))),
                  produit=cfg.PRODUIT, version=cfg.VERSION, unite=cfg.UNITE, alpha=cfg.ALPHA, z_seuil=cfg.Z_SEUIL,
                  fin_pre=cfg.FIN_PRE[:4] if cfg.FIN_PRE else None, nom_coupure=cfg.NOM_COUPURE,
                  genere_le=date.today().isoformat(), debut=ctx.etiq[0], fin=ctx.etiq[-1]),
        etiq=ctx.etiq, hunga=ctx.evenement, evenement=ctx.evenement, npre=npre,
        lat=liste(lat, 2), lon=liste(lon, 2),
        niveaux=[niveau], niveau_defaut=0, profils=[],
        cycle=dict(mois=list(range(1, 13)), moyenne=liste(clim["mean"].values), ecart_type=liste(clim["std"].values)),
    )
    sortie = cfg.SITE / "data" / f"donnees_{cfg.NOM}.js"
    sortie.write_text("window.DONNEES = " + json.dumps(donnees, ensure_ascii=False, separators=(",", ":")) + ";\n",
                      encoding="utf-8")
    print(f"{sortie}  ({sortie.stat().st_size / 1e6:.2f} Mo)")
    for f in cfg.SORTIE_ZONE.glob("*.csv"):
        shutil.copy(f, cfg.SITE / "data" / "csv" / f"{cfg.NOM}_{f.name}")
    print("CSV copiés dans", cfg.SITE / "data" / "csv")


main()

## Télécharger les résultats

In [ ]:
import shutil
import config as cfg
from google.colab import files
shutil.make_archive(f"resultats_{cfg.NOM}_maroc", "zip", cfg.SORTIE_ZONE)
files.download(f"resultats_{cfg.NOM}_maroc.zip")
files.download(str(cfg.SITE / "data" / f"donnees_{cfg.NOM}.js"))